# ROW Deep Learning Land Classification

## Project Overview

This notebook performs ROW easement classification by combining:
- NDVI-based vegetation classification
- Pretrained deep learning land cover model

The workflow:
1. Clip raster to easement polygon
2. Compute NDVI
3. Run deep learning classification
4. Fuse results
5. Export final raster

## 1. Environment Setup & Verification


Verify active Python environment and installed dependencies before running analysis.

In [43]:
import sys
print("Current Python executable:")
print(sys.executable)

print("\n\nAvailable conda environments:")
!conda info --envs

print("\nInstalled packages in the active environment:")
!conda list

Current Python executable:
C:\Program Files\ArcGIS\Pro\bin\ArcGISPro.exe


Available conda environments:
# conda environments:
#
base                     C:\Program Files\ArcGIS\Pro\bin\Python
arcgispro-py3            C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3
arcgispro-clean       *  C:\Users\45715\AppData\Local\ESRI\conda\envs\arcgispro-clean


Installed packages in the active environment:
# packages in environment at C:\Users\45715\AppData\Local\ESRI\conda\envs\arcgispro-clean:
#
# Name                    Version                   Build  Channel
_py-xgboost-mutex         2.0                       cpu_0  
_tflow_select             2.5.0                       gpu    esri
absl-py                   1.3.0            py37haa95532_0  
addict                    2.4.0                      py_4    esri
alembic                   1.6.4              pyhd3eb1b0_0  
appdirs                   1.4.4                      py_0  
arcgis                    1.9.1                 py37_2327 

laspy                     1.7.0                    py37_1    esri
lerc                      2.2                        py_0    esri
libboost                  1.73.0              h6c2663c_11  
libbrotlicommon           1.2.0                         0    esri
libbrotlidec              1.2.0                         0    esri
libbrotlienc              1.2.0                         0    esri
libdeflate                1.8                  h2bbff1b_5  
libopencv                 4.5.2           py37_cuda11.1_cudnn8.1_7    esri
libpng                    1.6.37               h2a8f88b_0  
libprotobuf               3.13.0.1             h200bbdf_0  
libsodium                 1.0.18                        2    esri
libtiff                   4.3.0                         1    esri
libuv                     1.51.0                        0    esri
libwebp                   1.2.2                h2bbff1b_0  
libxgboost                1.5.1                hd77b12b_0  
libxml2                   2.9.12     

pywin32-security          228                      py37_3    esri
pywinpty                  0.5.7                    py37_0    esri
pyyaml                    5.4.1            py37h2bbff1b_1  
pyzmq                     22.2.1                   py37_0    esri
rdflib                    5.0.0                    py37_0    esri
regex                     2021.8.3         py37h2bbff1b_0  
requests                  2.25.1             pyhd3eb1b0_0  
requests-kerberos         0.12.0                        0    esri
requests-negotiate-sspi   0.5.2                    py37_1    esri
requests-oauthlib         1.3.0                      py_0  
requests-toolbelt         0.9.1                      py_0  
requests_ntlm             1.1.0                      py_0    esri
retrying                  1.3.3                    py37_2  
rsa                       4.7.2              pyhd3eb1b0_1  
sacremoses                0.0.43                     py_0    esri
saspy                     3.7.3                     

## 2. Class Definitions

In [44]:
# Class dictionary
CLASSES = {
    "WATER": 0,
    "WETLANDS": 1,
    "TREE CANOPY": 2,
    "SHRUBLAND": 3,
    "LOW VEGETATION": 4,
    "BARREN": 5,
    "STRUCTURES": 6,
    "IMPERVIOUS SURFACES": 7,
    "IMPERVIOUS ROADS": 8,
    "NO DATA": 255
}

# Aliases for readability
WATER = CLASSES["WATER"]
WETLANDS = CLASSES["WETLANDS"]
TREE_CANOPY = CLASSES["TREE CANOPY"]
SHRUBLAND = CLASSES["SHRUBLAND"]
LOW_VEGETATION = CLASSES["LOW VEGETATION"]
BARREN = CLASSES["BARREN"]
STRUCTURES = CLASSES["STRUCTURES"]
IMPERVIOUS_SURFACES = CLASSES["IMPERVIOUS SURFACES"]
IMPERVIOUS_ROADS = CLASSES["IMPERVIOUS ROADS"]
NO_DATA = CLASSES["NO DATA"]

## 3. Helper Functions

### 3.1 NDVI & DL Functions

In [45]:
def compute_ndvi_and_classify(input_raster, show_visual):
    raster = arcpy.Raster(input_raster)
    
    # Convert to NumPy for pixel-wise NDVI computation
    arr = arcpy.RasterToNumPyArray(raster)
    
    # Define variables for each of the 4 bands (Red, Green, Blue, & Infrared) and the no-data mask
    r = arr[0].astype(float)
    g = arr[1].astype(float)
    b = arr[2].astype(float)
    nir = arr[3].astype(float)
    
    # Build NoData mask
    nodata_mask = (r == raster.noDataValue)
        
    # Compute NDVI for each pixel
    ndvi = (nir - r) / (nir + r + 1e-5)
    ndvi = np.nan_to_num(ndvi, nan=0)
        
    # Classify each pixel
    ndvi_class = np.zeros(ndvi.shape, dtype=np.uint8)
    ndvi_class[ndvi < 0.2] = IMPERVIOUS_ROADS                    # Roads / built
    ndvi_class[(ndvi >= 0.2) & (ndvi < 0.5)] = LOW_VEGETATION    # Grass / low vegetation
    ndvi_class[ndvi >= 0.5] = TREE_CANOPY                        # Trees / dense vegetation
                       
    # Apply NoData Mask
    ndvi_class[nodata_mask] = NO_DATA
    
    print("NDVI classes:", np.unique(ndvi_class))
    
    # Visualize NDVI
    if(show_visual):
        import matplotlib.pyplot as plt
        plt.imshow(ndvi, cmap="RdYlGn")
        plt.colorbar()
        plt.title("NDVI")
        plt.show()

    return ndvi, ndvi_class



def run_dl_model(input_raster, model_path):
    raster = arcpy.Raster(input_raster)
    
   
    # Extract RGB only (bands 1–3) for input into DL model
    rgb_raster = arcpy.management.CompositeBands(
        [
            f"{raster}/Band_1",
            f"{raster}/Band_2",
            f"{raster}/Band_3"
        ],
        "in_memory/rgb_composite"
    )
    
    rgb_raster = arcpy.Raster(rgb_raster)

    print(arcpy.Raster(rgb_raster).bandCount)

    result = ClassifyPixelsUsingDeepLearning(rgb_raster, model_path)
    return result



def fuse_results(ndvi_class, dl_class):
    final = dl_class.copy()
    
    # NDVI mask encompassing all vegetation pixels
    veg_mask = (
        (ndvi_class == LOW_VEGETATION) |
        (ndvi_class == TREE_CANOPY)
    )

    # DL mask encompassing all non-vegetation pixels
    NON_VEG_CLASSES = [IMPERVIOUS_ROADS, IMPERVIOUS_SURFACES, STRUCTURES, BARREN]
    nonveg_dl_mask = np.isin(dl_class, NON_VEG_CLASSES)
    
    # Only fix pixels where DL got vegetation wrong
    correction_mask = veg_mask & nonveg_dl_mask

    # Assign NDVI vegetation class
    final[correction_mask] = ndvi_class[correction_mask]
    
    # Preserve no-data pixels
    nodata_mask = (ndvi_class == NO_DATA)
    final[nodata_mask] = NO_DATA
    
    print("Final classes:", np.unique(final))

    return final

### 3.2 Raster Input Functions

In [99]:
def get_polygon(pipeline_line, easement_points, output_fc):
    # Get polygon based on pipeline and closest easement point
    print("Buffering pipeline selection...")
    
    # Find closest easement point to specified pipeline
    arcpy.analysis.Near(
        in_features=pipeline_line, 
        near_features=easement_points
    )
    
    # Define default WIDTH
    width = 20  # default = 20 yards (60 feet)
    
    
    # Get object ID (NEAR_FID) of closest easement point
    near_fid = None
    with arcpy.da.SearchCursor(pipeline_line, ["NEAR_FID"]) as cursor:
        for row in cursor:
            near_fid = row[0]

    # Get WIDTH from closest easement point
    where_clause = f"OBJECTID = {near_fid}"
    
    with arcpy.da.SearchCursor(
        easement_points,
        ["WIDTH"],
        where_clause
    ) as cursor:
        for row in cursor:
            width = row[0]
    
    print(f"Using easement width: {width} yards")
    
    # Buffer pipeline line to get polygon output
    polygon = arcpy.analysis.Buffer(
        in_features=pipeline_line,
        out_feature_class=output_fc,
        buffer_distance_or_field=f"{width} Yards",
        line_side="FULL",
        line_end_type="FLAT",
        dissolve_option="ALL"
    )

    return polygon

def clip_raster_to_polygon(active_map, raster_path, input_polygon):
    # Specify active map for sourcing raster layer
    m = active_map

    # Get raster path
    input_raster = None
    for lyr in m.listLayers():
        if lyr.name == raster_path:
            input_raster = arcpy.Raster(lyr.dataSource)

    if input_raster is None:
        raise ValueError(f"Layer '{raster_path}' not found")
        
    # Clip the inputted polygon to the NAIP raster imagery
    print("Clipping raster to polygon extent...")
    clipped_raster = arcpy.management.Clip(
        input_raster,
        "#",
        "in_memory/temp_clip",
        input_polygon,
        "#",
        "ClippingGeometry"
    )

    return clipped_raster

### 3.3 Raster Output Functions

In [63]:
def save_raster(input_raster_path, raster_classifications, symbology_dir, output_dir):
    # Preserve and save output as spatial reference (based on the input's spatial data)
    print("Saving output raster...")
    os.makedirs(output_dir, exist_ok=True)
    
    raster_obj = arcpy.Raster(input_raster_path)

    out_raster = arcpy.NumPyArrayToRaster(
        raster_classifications,
        lower_left_corner=raster_obj.extent.lowerLeft,
        x_cell_size=raster_obj.meanCellWidth,
        y_cell_size=raster_obj.meanCellHeight,
        value_to_nodata=NO_DATA
    )
    
    # Assign projection (coordinates)
    arcpy.DefineProjection_management(out_raster, raster_obj.spatialReference)

    # Save raster
    output_path = os.path.join(output_dir, "final_classification.tif")
    out_raster.save(output_path)
    
    # --- Add to map ---
    aprx = arcpy.mp.ArcGISProject("CURRENT")
    m = aprx.listMaps("Map2")[0]

    m.addDataFromPath(output_path)

    # Get the newly added layer (last one in map)
    added_layer = m.listLayers()[-1]

    # Apply symbology to saved raster
    symbology_path = os.path.join(symbology_dir, "final_classification_symbology.lyrx")
    arcpy.management.ApplySymbologyFromLayer(added_layer, symbology_path)
    
    print("Save complete")
    
    return output_path

## 4. Easement Analysis Pipeline

In [64]:
def analyze_easement(clipped_raster, model_path):
    # Run NDVI pipeline (NumPy)
    print("Running NDVI pipeline...")
    ndvi, ndvi_class = compute_ndvi_and_classify(clipped_raster, show_visual=True)

    # Run deep learning model (ArcPy)
    print("Running deep learning model...")
    dl_raster = run_dl_model(clipped_raster, model_path)
#     dl_class[ndvi_class == NO_DATA] = NO_DATA

    # Convert raster to numPy array
    print("Converting DL output to NumPy...")
    dl_array = arcpy.RasterToNumPyArray(
        arcpy.Raster(dl_raster), 
        nodata_to_value=NO_DATA
    )

    # Fuse results
    print("Fusing NDVI and DL results...")
    final_classifications = fuse_results(ndvi_class, dl_array)
    
    print("Easement analysis complete")
    return final_classifications

## 5. Main Execution

### 5.1 Imports & Setup

In [65]:
# Imports
print("Starting imports...")
import os

Starting imports...


In [66]:
import arcpy

In [67]:
import numpy as np

In [68]:
from arcgis.gis import GIS

In [69]:
from arcpy.sa import ExtractByMask

In [70]:
from arcpy.ia import ClassifyPixelsUsingDeepLearning

In [71]:
# Global setting
arcpy.env.overwriteOutput = True

print("Imports complete")

Imports complete


### 5.2 Main Function

In [95]:
def main():
    # Connect
    print("Starting connect to GIS...")
    gis = GIS("home")
    print("GIS connection complete")
    
    # Directory structure
    aprx = arcpy.mp.ArcGISProject("CURRENT")
    project_dir = os.path.dirname(aprx.filePath)
    data_dir = os.path.join(project_dir, "data")
    dl_model_dir = os.path.join(project_dir, "dl_model")
    symbology_dir = os.path.join(project_dir, "symbology")
    output_dir = os.path.join(project_dir, "outputs")
    
    # Map & Layer Structure
    m = aprx.listMaps("Map2")[0]
    buffer_fc = os.path.join("in_memory", "easement_buffer")
    input_line_name = "pipeline_test"
    easement_points_name = "Easement_Points"
    raster_path_name = "NAIP_USDA_CONUS_PRIME"

    # Get pipeline line and easement point layer locations in active map
    input_line = None
    easement_points = None
    for lyr in m.listLayers():
        if lyr.name == input_line_name:
            input_line = lyr.dataSource
        if lyr.name == easement_points_name:
            easement_points = lyr.dataSource

    if input_line is None:
        raise ValueError(f"Layer '{input_line_name}' not found")
    if easement_points is None:
        raise ValueError(f"Layer '{easement_points_name}' not found")
    
    # Get polygon based on pipeline input
    polygon = get_polygon(input_line, easement_points, buffer_fc)
    
    # Clip raster to pipeline polygon
    clipped_raster = clip_raster_to_polygon(m, raster_path_name, polygon)

    # Get deep learning model package path
    model_path = os.path.join(dl_model_dir, "HighResolutionLandCoverClassification_USA.dlpk")
    
    # Run easement analysis pipeline
    raster_classifications = analyze_easement(clipped_raster, model_path)
    
    # Save classified raster to notebook directory
    output_raster_path = save_raster(raster_path, raster_classifications, symbology_dir, output_dir)
    
    return output_raster_path

### 5.3 Run Main

In [100]:
# Run main
if __name__ == "__main__":
    main()

Starting connect to GIS...
GIS connection complete
Buffering pipeline selection...
Using easement width: 10.0 feet
Clipping raster to polygon extent...
Running NDVI pipeline...
NDVI classes: [  8 255]
Running deep learning model...
3


Traceback (most recent call last):
  File "C:\Users\45715\AppData\Local\Temp\ArcGISProTemp22948\HighResolutionLandCoverClassification_USA.dlpk\HighResolutionLandCoverClassification_USA_ArcGISImageClassifier.py", line 174, in initialize
    self.child_image_classifier.initialize(model, model_as_file)
  File "C:\Users\45715\AppData\Local\Temp\ArcGISProTemp22948\HighResolutionLandCoverClassification_USA.dlpk\_unet.py", line 243, in initialize
    self.unet = UnetClassifier.from_emd(data=None, emd_path=model)
  File "C:\Users\45715\AppData\Local\ESRI\conda\envs\arcgispro-clean\lib\site-packages\arcgis\learn\models\_unet.py", line 402, in from_emd
    return cls(data, **model_params, pretrained_path=str(model_file))
  File "C:\Users\45715\AppData\Local\ESRI\conda\envs\arcgispro-clean\lib\site-packages\arcgis\learn\models\_unet.py", line 223, in __init__
    split_on=backbone_split,
  File "C:\Users\45715\AppData\Local\ESRI\conda\envs\arcgispro-clean\lib\site-packages\fastai\vision\learner.p

ExecuteError: Failed to execute. Parameters are not valid.
ERROR 002667: Unable to initialize python raster function with scalar arguments. [C:\Users\45715\AppData\Local\Temp\ArcGISProTemp22948\HighResolutionLandCoverClassification_USA.dlpk\HighResolutionLandCoverClassification_USA_ArcGISImageClassifier.py]
Unable to initialize python raster function with scalar arguments.
Failed to execute (ClassifyPixelsUsingDeepLearning).
